# Portfolio Project: King County House Price Modeling

This notebook documents an end-to-end regression workflow on residential sales data from King County, WA. The focus is practical model building: data quality checks, feature exploration, baseline models, and regularized refinements.

## Roadmap

1. Data ingestion and schema validation
2. Data wrangling and missing-value handling
3. Exploratory analysis and signal checks
4. Baseline linear models
5. Evaluation and refinement with regularization

## Project Context

Goal: estimate house sale price from structural, location, and quality features.

Approach: start with lightweight inspection and cleaning, then move through EDA into progressively stronger regression baselines.

## Dataset Snapshot

This dataset contains residential home sales in King County (Seattle region), covering transactions between May 2014 and May 2015. Source data was adapted from Kaggle for educational use.

| Variable      | Description                                                                                                 |
| ------------- | ----------------------------------------------------------------------------------------------------------- |
| id            | A notation for a house                                                                                      |
| date          | Date house was sold                                                                                         |
| price         | Price is prediction target                                                                                  |
| bedrooms      | Number of bedrooms                                                                                          |
| bathrooms     | Number of bathrooms                                                                                         |
| sqft_living   | Square footage of the home                                                                                  |
| sqft_lot      | Square footage of the lot                                                                                   |
| floors        | Total floors (levels) in house                                                                              |
| waterfront    | House which has a view to a waterfront                                                                      |
| view          | Has been viewed                                                                                             |
| condition     | How good the condition is overall                                                                           |
| grade         | overall grade given to the housing unit, based on King County grading system                                |
| sqft_above    | Square footage of house apart from basement                                                                 |
| sqft_basement | Square footage of the basement                                                                              |
| yr_built      | Built Year                                                                                                  |
| yr_renovated  | Year when house was renovated                                                                               |
| zipcode       | Zip code                                                                                                    |
| lat           | Latitude coordinate                                                                                         |
| long          | Longitude coordinate                                                                                        |
| sqft_living15 | Living room area in 2015(implies-- some renovations) This might or might not have affected the lotsize area |
| sqft_lot15    | LotSize area in 2015(implies-- some renovations)                                                            |


## Environment Setup

In [ ]:
# All Libraries required for this lab are listed below. The libraries pre-installed on Skills Network Labs are commented.
# !mamba install -qy pandas==1.3.4 numpy==1.21.4 seaborn==0.9.0 matplotlib==3.5.0 scikit-learn==0.20.1
# Note: If your environment doesn't support "!mamba install", use "!pip install"

In [ ]:
# Surpress warnings:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

In [ ]:
#!pip install -U scikit-learn

In [ ]:
import piplite
await piplite.install('seaborn')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,PolynomialFeatures
from sklearn.linear_model import LinearRegression
%matplotlib inline

# Phase 1: Data Ingestion

Fetch the dataset and persist it locally for repeatable runs.

In [ ]:
from pyodide.http import pyfetch

async def download(url, filename):
    response = await pyfetch(url)
    if response.status == 200:
        with open(filename, "wb") as f:
            f.write(await response.bytes())

In [ ]:
filepath='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/FinalModule_Coursera/data/kc_house_data_NaN.csv'

In [ ]:
await download(filepath, "housing.csv")
file_name="housing.csv"

Load the CSV into a pandas DataFrame.

In [ ]:
df = pd.read_csv(file_name)

Environment note: in browser-based JupyterLite, the file is downloaded first and then loaded from local storage. In a local Python environment, the source URL can be passed directly to `pd.read_csv`.

In [ ]:
#filepath='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/FinalModule_Coursera/data/kc_house_data_NaN.csv'
#df = pd.read_csv(filepath, header=None)

Quick sanity check: inspect the first few rows.

In [ ]:
df.head()

### Checkpoint 1: Schema audit

Review column data types before feature selection and modeling.

In [ ]:
# Checkpoint 1: inspect inferred dtypes
df.dtypes

Generate a statistical summary to understand ranges, central tendency, and potential outliers.

In [ ]:
df.describe()

# Phase 2: Data Quality and Wrangling

### Checkpoint 2: Drop non-model fields and re-profile

Remove identifier-only columns (`id`, `Unnamed: 0`) and regenerate summary stats.

In [ ]:
# Checkpoint 2: drop non-model columns and re-profile
df.drop(columns=['id', 'Unnamed: 0'], inplace=True)
df.describe()

Missing values are present in `bedrooms` and `bathrooms`; quantify impact before imputation.

In [ ]:
print("number of NaN values for the column bedrooms :", df['bedrooms'].isnull().sum())
print("number of NaN values for the column bathrooms :", df['bathrooms'].isnull().sum())


Impute missing `bedrooms` values with the column mean as a baseline strategy.

In [ ]:
mean=df['bedrooms'].mean()
df['bedrooms'].replace(np.nan,mean, inplace=True)

Apply the same mean-imputation baseline to missing `bathrooms` values.

In [ ]:
mean=df['bathrooms'].mean()
df['bathrooms'].replace(np.nan,mean, inplace=True)

In [ ]:
print("number of NaN values for the column bedrooms :", df['bedrooms'].isnull().sum())
print("number of NaN values for the column bathrooms :", df['bathrooms'].isnull().sum())

# Phase 3: Exploratory Analysis

### Checkpoint 3: Floor-level distribution

Quantify listing counts by floor count to understand housing inventory shape.

In [ ]:
# Checkpoint 3: floor-count distribution
df['floors'].to_frame().value_counts().to_frame()

### Checkpoint 4: Waterfront vs. price spread

Use a boxplot to compare price distribution and outlier behavior by waterfront flag.

In [ ]:
sns.boxplot(df, x='waterfront', y='price')
plt.ylim(0,)

### Checkpoint 5: `sqft_above` signal check

Use a regression plot to validate direction and strength of association with target price.

In [ ]:
# Checkpoint 5: sqft_above vs price trend
sns.regplot(df, x='sqft_above', y='price')
plt.ylim(0,)

Compute full numeric correlation against `price` to identify high-signal candidate features.

In [ ]:
df_numeric = df.select_dtypes(include=[np.number])
df_numeric.corr()['price'].sort_values()

# Phase 4: Baseline Modeling

Start with a single-feature linear baseline using longitude (`long`) and record in-sample $R^2$.

In [ ]:
X = df[['long']]
Y = df['price']
lm = LinearRegression()
lm.fit(X,Y)
lm.score(X, Y)

### Checkpoint 6: Single-feature baseline (`sqft_living`)

Train linear regression on `sqft_living` and log in-sample $R^2$.

In [ ]:
# Checkpoint 6: single-feature linear baseline
lm_sqft_living = LinearRegression()
X=df[['sqft_living']]
Y=df['price']
lm_sqft_living.fit(X, Y)
lm_sqft_living.score(X, Y)

### Checkpoint 7: Multi-feature linear baseline

Fit linear regression using a curated feature list and compare $R^2$ to prior baselines.

In [ ]:
features =["floors", "waterfront","lat" ,"bedrooms" ,"sqft_basement" ,"view" ,"bathrooms","sqft_living15","sqft_above","grade","sqft_living"]     

Evaluate in-sample $R^2$ for the multi-feature linear model.

In [ ]:
# Checkpoint 7: multi-feature linear baseline
X=df[features]
Y=df['price']
lm=LinearRegression()
lm.fit(X, Y)
lm.score(X, Y)

Pipeline components for nonlinear expansion + scaling + linear estimation:

- `StandardScaler()`
- `PolynomialFeatures(include_bias=False)`
- `LinearRegression()`

In [ ]:
Input=[('scale',StandardScaler()),('polynomial', PolynomialFeatures(include_bias=False)),('model',LinearRegression())]

### Checkpoint 8: Polynomial pipeline baseline

Build the pipeline from the components above, fit on the selected features, and record $R^2$.

In [ ]:
# Checkpoint 8: polynomial pipeline baseline
from sklearn.metrics import r2_score
pipe = Pipeline(Input)
pipe.fit(X, Y)
r2_score(pipe.predict(X), Y)

# Phase 5: Evaluation and Refinement

Import evaluation and splitting utilities for holdout testing.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
print("done")

Create train/test splits to separate fitting from performance validation.

In [ ]:
features =["floors", "waterfront","lat" ,"bedrooms" ,"sqft_basement" ,"view" ,"bathrooms","sqft_living15","sqft_above","grade","sqft_living"]    
X = df[features]
Y = df['price']

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.15, random_state=1)


print("number of test samples:", x_test.shape[0])
print("number of training samples:",x_train.shape[0])

### Checkpoint 9: Ridge regression on holdout

Fit Ridge with `alpha=0.1` on training data and evaluate test-set $R^2$.

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
# Checkpoint 9: Ridge on holdout split
ridge = Ridge(alpha=0.1)
ridge.fit(x_train, y_train)
ridge.score(x_test, y_test)

### Checkpoint 10: Polynomial features + Ridge

Apply a second-order polynomial transform, retrain Ridge (`alpha=0.1`), and compare test-set $R^2$ against the linear Ridge baseline.

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
x_train_poly = poly.fit_transform(x_train)
x_test_poly = poly.transform(x_test)

ridge_poly = Ridge(alpha=0.1)
ridge_poly.fit(x_train_poly, y_train)
ridge_poly.score(x_test_poly, y_test)

## Wrap-up

This notebook captures a practical progression from raw data ingestion to regularized nonlinear regression, with each checkpoint recording model behavior and trade-offs.

## Source Attribution

Original exercise structure adapted from IBM Skills Network material; content and narrative rewritten for portfolio presentation.

Dataset: King County House Sales (Kaggle-derived educational variant).

Notebook revision: converted assignment-style prompts into engineering checkpoints for portfolio use.